# Week 3 · Support vector machines

An SVM is a linear classifier, like logistic regression, with a different rule for choosing the line: **of all the lines that separate the classes, take the one with the widest gap around it.** That one idea gives the margin, the support vectors, the hinge loss and, through the dual problem, the kernel trick. Labels here are $y \in \{-1, +1\}$ instead of $\{0, 1\}$, because it makes the formulas shorter. **Reading:** ISL Ch. 9; StatQuest "Support Vector Machines" parts 1–3.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **margin**: The empty band ("street") around the decision line, out to the nearest points. The SVM makes it as wide as possible.
- **support vectors**: The training points that touch the edge of the margin, or are inside it. Only they determine the line.
- **slack $\xi_i$**: How far point $i$ violates the margin. 0 for points safely outside it.
- **hinge loss**: $\max(0,\ 1 - y z)$: zero for points beyond the margin on the right side, growing linearly otherwise.
- **kernel**: A function $k(x, x')$ that equals a dot product $\phi(x)^\top \phi(x')$ in some bigger feature space, computed without ever building $\phi$.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from figures import *
import checks; checks.load(globals(), "05")      # defines check(n)

## 0. Recap first · 10 min, closed book

**Quick check:** A node holds 3 defective and 1 OK part. Its Gini impurity?

<details><summary>Answer</summary>

$2 \times \frac{3}{4} \times \frac{1}{4} = 0.375$, or $1 - (0.75^2 + 0.25^2) = 0.375$. [04 §2 ↗](04_decision_trees.ipynb)

</details>

**Quick check:** Split a parent with Gini 0.5 into a left child of 4 samples (Gini 0) and a right child of 6 samples (Gini 0.444). Impurity decrease?

<details><summary>Answer</summary>

$0.5 - (0.4 \times 0 + 0.6 \times 0.444) = 0.5 - 0.267 = 0.233$. [04 §3 ↗](04_decision_trees.ipynb)

</details>

**Quick check:** Why do trees need no feature scaling, while kNN does?

<details><summary>Answer</summary>

A tree split compares values **within one feature** ("temp ≤ 72"), so units don't matter. kNN adds differences **across** features, so a feature with big numbers dominates. [Week 2 · 05 §5 ↗](../week2/05_logistic_regression_knn.ipynb)

</details>

**Quick check:** Projection of $b$ onto the direction $a$: what's the formula for its **length**?

<details><summary>Answer</summary>

$\frac{a \cdot b}{\lVert a \rVert}$ (the full projection vector is $\frac{a \cdot b}{a \cdot a}\, a$). You'll use it in section 2. [Week 2 · 01 §4 ↗](../week2/01_linear_algebra_review.ipynb)

</details>

**Quick check:** `one_hot(np.array([2, 0]), 3)` returns ...?

<details><summary>Answer</summary>

`[[0, 0, 1], [1, 0, 0]]`: rows 2 and 0 of `np.eye(3)`. [02, exercise 15 ↗](02_knn_and_numpy_workout.ipynb)

</details>

## 1. The idea: the widest street

**A support vector machine separates two classes with the line that stays as far as possible from the closest points of both classes.**

🖼 **Picture.** Six points and three lines that all separate them perfectly. A and B pass very close to some points; C stays far from both groups.

In [ ]:
fig_many_separators()

Logistic regression would accept any of these lines once the loss is small. The SVM prefers **C**. Why? A new point near the edge of the blue group is more likely to land on the wrong side of A than of C. A wide gap is a safety buffer against noise, and in theory it also comes with better guarantees on test error.

**The setting.** Training points $x_i$ with labels $y_i \in \{-1, +1\}$, a line $w^\top x + b = 0$ and the score $z = w^\top x + b$. The prediction is the sign of $z$. Correct means $y_i z_i > 0$: both have the same sign. That's why the $\pm 1$ labels are handy.

## 2. The distance from a point to a line

**The signed distance from a point $x$ to the line $w^\top x + b = 0$ is its score divided by the length of $w$:**

$$
\text{dist}(x) = \frac{w^\top x + b}{\lVert w \rVert}
$$

**Where it comes from: a projection.** Pick any point $x_0$ on the line, so $w^\top x_0 = -b$. The distance from $x$ to the line is the length of $x - x_0$ **projected onto the direction perpendicular to the line**, and that direction is $w$ ([01 §2 ↗](01_logistic_regression_in_pictures.ipynb)). The length of a projection onto $w$ is $\frac{w^\top v}{\lVert w \rVert}$ ([Week 2 · 01 §4 ↗](../week2/01_linear_algebra_review.ipynb)):

$$
\text{dist}(x) = \frac{w^\top (x - x_0)}{\lVert w \rVert} = \frac{w^\top x - w^\top x_0}{\lVert w \rVert} = \frac{w^\top x - (-b)}{\lVert w \rVert} = \frac{w^\top x + b}{\lVert w \rVert}
$$

**Worked example.** The point $(1, 1)$ and the line $x_1 + x_2 = 4.5$, i.e. $w = (1, 1)$ and $b = -4.5$: the score is $1 + 1 - 4.5 = -2.5$, $\lVert w \rVert = \sqrt{2} = 1.414$, and the distance is $\frac{-2.5}{1.414} = -1.768$ (negative: the $-1$ side). The same line written as $w = (\frac{2}{3}, \frac{2}{3})$, $b = -3$ gives the score $-1.667$ and $\lVert w \rVert = 0.943$, so again $-1.768$.

**Key property: rescaling $w$ and $b$ together doesn't change the line or the distances; it only changes the scores.** The SVM uses this freedom in the next section.

## 3. The hard-margin problem

**Rescale $w$ and $b$ so that the closest points have score exactly $\pm 1$. Then the street is $\frac{2}{\lVert w \rVert}$ wide, and maximizing its width is the same as minimizing $\lVert w \rVert$.**

**Step 1, fix the scale.** Since rescaling is free (section 2), choose it so that $y_i (w^\top x_i + b) = 1$ for the points closest to the line. Every other point is further away, so

$$
y_i\,(w^\top x_i + b) \ge 1 \quad \text{for every } i
$$

**Step 2, measure the street.** A closest point has score $\pm 1$, so its distance to the line is $\frac{1}{\lVert w \rVert}$ (section 2). The edges of the street are the two lines $w^\top x + b = \pm 1$, so the total width is $\frac{2}{\lVert w \rVert}$.

**Step 3, the optimization problem.** Making $\frac{2}{\lVert w \rVert}$ as large as possible is the same as making $\lVert w \rVert$ as small as possible. Squaring it and putting $\frac{1}{2}$ in front only makes the calculus cleaner:

$$
\min_{w,\, b}\ \frac{1}{2}\lVert w \rVert^2 \quad \text{subject to} \quad y_i\,(w^\top x_i + b) \ge 1 \ \text{ for all } i
$$

**Worked example, by hand.** Negatives $(1, 1)$, $(2, 1)$, $(1, 2)$; positives $(3, 3)$, $(4, 3)$, $(3, 4)$.

- The two groups are mirror images across the diagonal, so the best line is perpendicular to $(1, 1)$: $w = c\,(1, 1)$, i.e. a line $x_1 + x_2 = \text{const}$.
- The closest negatives, $(2, 1)$ and $(1, 2)$, have $x_1 + x_2 = 3$. The closest positive, $(3, 3)$, has $x_1 + x_2 = 6$. The widest street runs between those two lines, and its middle is $x_1 + x_2 = 4.5$.
- Scores of $\pm 1$ at those points: $3c + b = -1$ and $6c + b = +1$. Subtracting gives $3c = 2$, so $c = \frac{2}{3}$ and $b = -1 - 2 = -3$.
- So $w = (\frac{2}{3}, \frac{2}{3})$, $b = -3$, $\lVert w \rVert = \frac{2}{3}\sqrt{2} = 0.943$, and the width is $\frac{2}{0.943} = 2.121$. Check: the distance between the lines $x_1 + x_2 = 3$ and $x_1 + x_2 = 6$ is $\frac{3}{\sqrt 2} = 2.121$.

| Point | $y$ | $z = \frac{2}{3}(x_1 + x_2) - 3$ | $yz$ | Constraint |
|---|---|---|---|---|
| (1, 1) | −1 | −1.667 | 1.667 | ≥ 1 ✓ (outside the street) |
| (2, 1) | −1 | −1.000 | 1.000 | = 1: **on the edge** |
| (1, 2) | −1 | −1.000 | 1.000 | = 1: **on the edge** |
| (3, 3) | +1 | +1.000 | 1.000 | = 1: **on the edge** |
| (4, 3) | +1 | +1.667 | 1.667 | ≥ 1 ✓ |
| (3, 4) | +1 | +1.667 | 1.667 | ≥ 1 ✓ |

🖼 **Picture.** scikit-learn's answer for the same six points: the line $x_1 + x_2 = 4.5$, the street edges (dashed), and the three circled **support vectors**.

In [ ]:
fig_margin()

In [ ]:
from sklearn.svm import SVC
X = np.array([[1, 1], [2, 1], [1, 2], [3, 3], [4, 3], [3, 4]], dtype=float)
y = np.array([-1, -1, -1, 1, 1, 1])
svm = SVC(kernel="linear", C=1e6).fit(X, y)          # a huge C = no violations allowed (hard margin)
print(svm.coef_.round(3), svm.intercept_.round(3))   # → [[0.667 0.666]] [-3.]
print(svm.support_)                                  # → [1 2 3]   rows (2, 1), (1, 2), (3, 3)

## 4. Support vectors

**Only the points on the edge of the street, the support vectors, determine the solution. Move any other point (without crossing into the street) and nothing changes.**

In the table above, only $(2, 1)$, $(1, 2)$ and $(3, 3)$ have $y z = 1$. If you moved $(1, 1)$ to $(0, -5)$, the best line would be exactly the same. Compare logistic regression: **every** point pulls, even confident ones a little ([01 §6 ↗](01_logistic_regression_in_pictures.ipynb)). For an SVM, points beyond the street pull with exactly zero force. Section 6 shows this as a loss.

Consequences: an SVM's solution is **sparse** (it's described by a few points), and the outlier points far behind their own class don't affect it at all.

## 5. The soft margin: allowing some violations

**Real data overlaps, so no street is empty. The soft-margin SVM lets points violate the margin, and pays $C$ for each unit of violation.**

$$
\min_{w,\, b,\, \xi}\ \frac{1}{2}\lVert w \rVert^2 + C \sum_{i=1}^n \xi_i \quad \text{subject to} \quad y_i\,(w^\top x_i + b) \ge 1 - \xi_i,\ \ \xi_i \ge 0
$$

$\xi_i$ ("xi", the **slack**) is how far point $i$ falls short of the margin. The smallest slack that satisfies the constraint is $\xi_i = \max(0,\ 1 - y_i z_i)$:

| $y_i z_i$ | $\xi_i$ | Where the point is |
|---|---|---|
| $\ge 1$ | 0 | outside the street, on its own side: no penalty |
| between 0 and 1 | between 0 and 1 | inside the street, but still on the right side |
| $< 0$ | $> 1$ | on the wrong side of the line: misclassified |

**$C$ is the trade-off knob.** A large $C$ makes violations expensive: a narrow street that bends to fit the training data (low bias, high variance). A small $C$ makes them cheap: a wide street with many points inside it (high bias, low variance). Choose $C$ with cross-validation.

🖼 **Picture.** The same overlapping data with three values of $C$. Small $C$: a wide street, and 42 points inside it are support vectors. Large $C$: a narrow street with only 7 support vectors.

In [ ]:
fig_soft_margin(Cs=(0.01, 1, 100))

## 6. The hinge loss: an SVM is a loss plus a penalty

**Put $\xi_i = \max(0, 1 - y_i z_i)$ back into the objective and the soft-margin SVM becomes an ordinary loss to minimize: the hinge loss plus an L2 penalty, which is ridge's penalty again.**

$$
\min_{w,\, b}\ \frac{1}{n}\sum_{i=1}^n \underbrace{\max\big(0,\ 1 - y_i (w^\top x_i + b)\big)}_{\text{hinge loss}} + \frac{\lambda}{2}\lVert w \rVert^2
$$

This is the soft-margin problem divided by $nC$, with $\lambda = \frac{1}{nC}$, so it has the same minimizer. Now an SVM and logistic regression are the same kind of thing: **a loss of the signed score $yz$, plus a penalty.** Only the loss differs:

🖼 **Picture.** Three losses as functions of $yz$. Logistic loss never quite reaches 0, so every point keeps pulling a little. Hinge is exactly 0 from $yz = 1$ on: confident correct points stop pulling entirely.

In [ ]:
fig_margin_losses()

**Worked example.** With $w = (\frac{2}{3}, \frac{2}{3})$ and $b = -3$, the hinge loss of a few points:

| Point | $y$ | $z$ | $yz$ | hinge $\max(0, 1 - yz)$ | Pulls? |
|---|---|---|---|---|---|
| (4, 4) | +1 | 2.333 | 2.333 | 0 | no: far on the right side |
| (3, 3) | +1 | 1.000 | 1.000 | 0 | no: exactly on the edge |
| (2.5, 2.5) | +1 | 0.333 | 0.333 | 0.667 | yes: inside the street |
| (2, 2.5) | −1 | 0.000 | 0.000 | 1.000 | yes: on the line |
| (2, 2) | +1 | −0.333 | −0.333 | 1.333 | yes: wrong side |

**The (sub)gradient.** For a point with $y_i z_i < 1$, the hinge is $1 - y_i (w^\top x_i + b)$, whose gradient with respect to $w$ is $-y_i x_i$; for other points it's 0. (At exactly $y_i z_i = 1$ the hinge has a corner; we treat that point as not pulling.) So gradient descent on the SVM is

$$
w \leftarrow w - \eta\Big(\lambda w - \frac{1}{n}\sum_{i:\ y_i z_i < 1} y_i\, x_i\Big), \qquad b \leftarrow b + \eta\, \frac{1}{n}\sum_{i:\ y_i z_i < 1} y_i
$$

Read it like [01 §6 ↗](01_logistic_regression_in_pictures.ipynb): each point inside the street or on the wrong side **pulls $w$ toward $y_i x_i$** with the same strength, every other point pulls with zero, and $\lambda w$ shrinks $w$ a little every step (which widens the street). You'll code this in exercise 4.

## 7. The dual problem: why only dot products matter

**The SVM problem has an equivalent "dual" form in which the data appear only through dot products $x_i^\top x_j$. The solution is a weighted sum of the support vectors, $w = \sum_i \alpha_i y_i x_i$.** This is what makes kernels possible.

**Where it comes from: Lagrange multipliers** (your optimization course). Give each constraint $y_i(w^\top x_i + b) - 1 \ge 0$ a multiplier $\alpha_i \ge 0$ and form the Lagrangian:

$$
\mathcal{L}(w, b, \alpha) = \frac{1}{2}\lVert w \rVert^2 - \sum_i \alpha_i \big[\, y_i (w^\top x_i + b) - 1 \,\big]
$$

**Set the derivatives to zero.** With respect to $w$: $w - \sum_i \alpha_i y_i x_i = 0$. With respect to $b$: $-\sum_i \alpha_i y_i = 0$. So

$$
w = \sum_i \alpha_i y_i x_i \qquad \text{and} \qquad \sum_i \alpha_i y_i = 0
$$

**Substitute them back.** $\frac{1}{2}\lVert w \rVert^2 = \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j y_i y_j\, x_i^\top x_j$. The constraint term expands to $\sum_i \alpha_i y_i\, w^\top x_i + b \sum_i \alpha_i y_i - \sum_i \alpha_i$. Here $\sum_i \alpha_i y_i\, w^\top x_i = \sum_{i,j} \alpha_i \alpha_j y_i y_j\, x_i^\top x_j$, and the $b$ term is 0. Putting it together:

$$
\max_{\alpha}\ \sum_i \alpha_i - \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j\, y_i y_j\, \underbrace{x_i^\top x_j}_{\text{only dot products}} \quad \text{subject to} \quad \alpha_i \ge 0,\ \ \sum_i \alpha_i y_i = 0
$$

(For the soft margin, the only change is $0 \le \alpha_i \le C$.) **Complementary slackness**, from the KKT conditions: $\alpha_i > 0$ only for points with $y_i z_i = 1$ exactly, i.e. the support vectors. Every other point has $\alpha_i = 0$ and drops out of $w = \sum_i \alpha_i y_i x_i$. That's section 4 again, now as algebra.

**Check with our six points.** scikit-learn reports $\alpha_i y_i$ for the support vectors: $-\frac{2}{9}$ for $(2, 1)$, $-\frac{2}{9}$ for $(1, 2)$ and $+\frac{4}{9}$ for $(3, 3)$. They sum to 0 ✓. And

$$
w = -\tfrac{2}{9}(2, 1) - \tfrac{2}{9}(1, 2) + \tfrac{4}{9}(3, 3) = \big(\tfrac{-4 - 2 + 12}{9},\ \tfrac{-2 - 4 + 12}{9}\big) = \big(\tfrac{2}{3}, \tfrac{2}{3}\big) \ \checkmark
$$

In [ ]:
print(svm.dual_coef_.round(3))                          # → [[-0.222 -0.222  0.444]]   α_i·y_i
print((svm.dual_coef_ @ svm.support_vectors_).round(3))   # → [[0.667 0.666]]          = w

**Why this matters:** a prediction is $w^\top x + b = \sum_i \alpha_i y_i\, (x_i^\top x) + b$. Training and predicting both need **only dot products between points**. Replace every dot product with a kernel, and you get a nonlinear SVM for free. That's next.

## 8. Kernels: a straight line in a bigger space

**A kernel $k(x, x')$ computes the dot product of two points after mapping them into a bigger feature space, $\phi(x)^\top \phi(x')$, without computing $\phi$. A straight line in that space becomes a curved boundary in the original one.**

🖼 **Picture.** Left: 1-D data that no single threshold can separate (orange in the middle). Right: add the feature $x^2$ and a horizontal line separates them.

In [ ]:
fig_lift_1d()

**The polynomial kernel, worked by hand.** In 2-D, $k(x, z) = (x^\top z + 1)^2$ equals $\phi(x)^\top \phi(z)$ with the 6 features

$$
\phi(x) = \big(1,\ \sqrt{2}x_1,\ \sqrt{2}x_2,\ x_1^2,\ \sqrt{2}x_1 x_2,\ x_2^2\big)
$$

Take $x = (1, 2)$ and $z = (3, 1)$. **The kernel way:** $x^\top z = 3 + 2 = 5$, so $(5 + 1)^2 = 36$: two multiplications, one addition, one square. **The long way:** $\phi(x) = (1, 1.414, 2.828, 1, 2.828, 4)$ and $\phi(z) = (1, 4.243, 1.414, 9, 4.243, 1)$; multiplying entry by entry gives $1 + 6 + 4 + 9 + 12 + 4 = 36$ ✓. Expanding $(x_1 z_1 + x_2 z_2 + 1)^2$ produces exactly those six products, which is why it works.

**The RBF (Gaussian) kernel** $k(x, z) = e^{-\gamma \lVert x - z \rVert^2}$ is 1 for identical points and falls toward 0 as they get far apart. Its $\phi$ has **infinitely many** features, so the kernel is the only way to use it. For $x = (1, 2)$, $z = (3, 1)$ and $\gamma = 0.5$: $\lVert x - z \rVert^2 = 4 + 1 = 5$, so $k = e^{-2.5} = 0.082$.

🖼 **Picture.** Circles inside circles: a linear SVM fails (accuracy near 0.5), while degree-2 polynomial and RBF kernels draw the circle.

In [ ]:
fig_kernels()

🖼 **Picture.** RBF with three values of $\gamma$. A small $\gamma$ means each point influences a wide area (smooth boundary). A large $\gamma$ draws tight blobs around single points: it overfits.

In [ ]:
fig_rbf_gamma(gammas=(0.1, 1, 30))

| Knob | Too small | Too large |
|---|---|---|
| $C$ | a wide street, many violations: underfits | a narrow street that fits noise: overfits |
| $\gamma$ (RBF) | almost linear: underfits | islands around single points: overfits |

## 9. SVMs in scikit-learn

**Always scale the features before an SVM.** The margin is measured in distances, so the kNN scaling problem applies here too ([Week 2 · 05 §5 ↗](../week2/05_logistic_regression_knn.ipynb)). Then tune $C$ and $\gamma$ together:

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
Xb, yb = load_breast_cancer(return_X_y=True)
Xa, Xt, ya, yt = train_test_split(Xb, yb, test_size=0.25, random_state=0, stratify=yb)
print(round(SVC().fit(Xa, ya).score(Xt, yt), 3))                                   # → 0.916   no scaling
print(round(make_pipeline(StandardScaler(), SVC()).fit(Xa, ya).score(Xt, yt), 3))  # → 0.958   scaled
grid = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [0.001, 0.01, 0.1, 1]}
search = GridSearchCV(make_pipeline(StandardScaler(), SVC()), grid, cv=5).fit(Xa, ya)
print(search.best_params_, round(search.best_score_, 3))   # → {'svc__C': 100, 'svc__gamma': 0.001} 0.988
print(round(search.score(Xt, yt), 3))                      # → 0.958

In a pipeline, a parameter's name is `step__parameter`, with two underscores; the step's name is the class name in lower case. $16$ combinations × 5 folds = 80 fits. A large $C$ with a tiny $\gamma$ is a nearly linear boundary that's allowed few violations, which suits this dataset.

**When to use an SVM.** Small-to-medium data (fitting gets slow beyond roughly $10^5$ samples) with a clear margin, and especially many features. Logistic regression gives probabilities directly; an SVM gives only a score (scikit-learn can calibrate it with `probability=True`).

## 10. Build it yourself

Labels are $\pm 1$ in exercises 2–4. `X` has shape `(n, d)`.

✏️ **Exercise 1 · Signed distance.** $\frac{Xw + b}{\lVert w \rVert}$ for every row. **Shape `(n,)`.**

In [ ]:
def signed_distance(X, w, b):
    pass  # your code

In [ ]:
check(1)

✏️ **Exercise 2 · Hinge loss.** $\frac{1}{n}\sum_i \max(0,\ 1 - y_i z_i) + \frac{\lambda}{2}\lVert w \rVert^2$. **Shape `()`.** `np.maximum(0, ...)` works element by element. (`np.max` would give the largest entry instead.)

In [ ]:
def hinge_loss(X, y, w, b, lam):
    pass  # your code

In [ ]:
check(2)

✏️ **Exercise 3 · Hinge subgradient.** Return `(grad_w, grad_b)`: $\lambda w - \frac{1}{n}\sum_{i \in A} y_i x_i$ and $-\frac{1}{n}\sum_{i \in A} y_i$, where $A$ is the set of points with $y_i z_i < 1$ (a mask). Shapes `(d,)` and `()`. Hint: `X[A].T @ y[A]` sums $y_i x_i$ over the rows in $A$.

In [ ]:
def hinge_grad(X, y, w, b, lam):
    pass  # your code

In [ ]:
check(3)

✏️ **Exercise 4 · Train a linear SVM.** Start from `w = np.zeros(d)`, `b = 0.0` and repeat `steps` times: compute `hinge_grad` **at the current `w`, `b`** (inside the loop: see 00 §A2), then step both. Return `(w, b)`. The checker reruns the same updates and compares.

In [ ]:
def svm_train(X, y, lam, lr, steps):
    pass  # your code

In [ ]:
check(4)

✏️ **Exercise 5 · Polynomial kernel matrix.** $K_{ij} = (a_i^\top b_j + c)^{\text{degree}}$ for all rows of `A` `(m, d)` and `B` `(n, d)`. **Shape `(m, n)`.** One matrix product does all the dot products at once.

In [ ]:
def poly_kernel(A, B, degree, c):
    pass  # your code

In [ ]:
check(5)

✏️ **Exercise 6 · RBF kernel matrix.** $K_{ij} = e^{-\gamma \lVert a_i - b_j \rVert^2}$. **Shape `(m, n)`.** The squared distances are your `distance_matrix` from 02, squared.

In [ ]:
def rbf_kernel(A, B, gamma):
    pass  # your code

In [ ]:
check(6)

✏️ **Exercise 7 · The feature map.** For 2-D inputs `X` `(n, 2)`, return $\phi(X)$ with the 6 columns of section 8, **shape `(n, 6)`**. The checker verifies that `phi2(A) @ phi2(B).T` equals `poly_kernel(A, B, 2, 1)`.

In [ ]:
def phi2(X):
    pass  # your code

In [ ]:
check(7)

In [ ]:
for n in range(1, 8):
    check(n)

---
Solutions: `solutions/05_solutions.py`. Next: `06_ensembles.ipynb`.